In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import string
 
# Load data
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [3]:
freq = df['answer'].value_counts()
print("Q1: Answer Frequency Distribution")
print(freq)
print(f"Most frequent: {freq.idxmax()} ({freq.max()})")
print(f"Least frequent: {freq.idxmin()} ({freq.min()})")
print(f"Sum of most + least frequent: {freq.max() + freq.min()}\n")

Q1: Answer Frequency Distribution
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most frequent: B (490)
Least frequent: E (324)
Sum of most + least frequent: 814



After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [4]:
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text
 
cleaned_prompts = df['prompt'].apply(clean_text)
all_words = []
for text in cleaned_prompts:
    all_words.extend(text.split())
vocab = set(all_words)
print("Q2: Vocabulary Size (cleaned prompts)")
print(f"Total unique words: {len(vocab)}\n")

Q2: Vocabulary Size (cleaned prompts)
Total unique words: 859



Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [5]:
row1 = df[df['id'] == 1].iloc[0]
cleaned_row1 = clean_text(row1['prompt'])
words_row1 = cleaned_row1.split()
filtered_row1 = [w for w in words_row1 if w not in ENGLISH_STOP_WORDS]
print("Q3: Words in Row ID 1 after stop word filtering")
print(f"Words remaining: {len(filtered_row1)}")
print(f"Words: {filtered_row1}\n")

Q3: Words in Row ID 1 after stop word filtering
Words remaining: 13
Words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']



Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [6]:
options = ['A', 'B', 'C', 'D', 'E']
 
combined_texts = []
for _, row in df.iterrows():
    combined = row['prompt'] + ' ' + ' '.join([str(row[o]) for o in options])
    combined_texts.append(combined)
 
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)
print("Q4: TF-IDF Vocabulary Size")
print(f"Total feature columns: {len(vectorizer.vocabulary_)}\n")

Q4: TF-IDF Vocabulary Size
Total feature columns: 2762



Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [7]:
row1_full = df[df['id'] == 1].iloc[0]
prompt_vec = vectorizer.transform([row1_full['prompt']])
optionA_vec = vectorizer.transform([str(row1_full['A'])])
sim = cosine_similarity(prompt_vec, optionA_vec)[0][0]
print("Q5: Cosine Similarity (Row ID 1, Prompt vs Option A)")
print(f"Similarity score: {round(sim, 4)}\n")

Q5: Cosine Similarity (Row ID 1, Prompt vs Option A)
Similarity score: 0.272



Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [8]:
correct = 0
for _, row in df.iterrows():
    prompt_v = vectorizer.transform([row['prompt']])
    sims = {}
    for opt in options:
        opt_v = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_v, opt_v)[0][0]
    best = max(sims, key=sims.get)
    if best == row['answer']:
        correct += 1
 
pct = correct / len(df) * 100
print("Q6: % Correct using Highest TF-IDF Similarity")
print(f"Percentage: {round(pct, 4)}%\n")

Q6: % Correct using Highest TF-IDF Similarity
Percentage: 13.55%



If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [9]:
def map_at_3(truth, preds):
    score = 0.0
    for i, p in enumerate(preds[:3]):
        if p == truth:
            score = 1.0 / (i + 1)
            break
    return score
 
print("Q7: MAP@3 (truth=C, preds=C A B)")
print(f"Score: {map_at_3('C', ['C','A','B'])}\n")

Q7: MAP@3 (truth=C, preds=C A B)
Score: 1.0



If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [10]:
print("Q8: MAP@3 (truth=B, preds=D B E)")
print(f"Score: {round(map_at_3('B', ['D','B','E']), 4)}\n")

Q8: MAP@3 (truth=B, preds=D B E)
Score: 0.5



The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [11]:
freq_sorted = df['answer'].value_counts()
top3 = list(freq_sorted.index[:3])
print("Q9: Majority Class Baseline")
print(f"Top 3 most frequent answers (prediction order): {top3}")
 
scores = [map_at_3(row['answer'], top3) for _, row in df.iterrows()]
print(f"MAP@3 Score: {round(np.mean(scores), 4)}\n")

Q9: Majority Class Baseline
Top 3 most frequent answers (prediction order): ['B', 'C', 'A']
MAP@3 Score: 0.4212



The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [12]:
tfidf_scores = []
for _, row in df.iterrows():
    prompt_v = vectorizer.transform([row['prompt']])
    sims = {}
    for opt in options:
        opt_v = vectorizer.transform([str(row[opt])])
        sims[opt] = cosine_similarity(prompt_v, opt_v)[0][0]
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    tfidf_scores.append(map_at_3(row['answer'], ranked))
 
print("Q10: TF-IDF Pipeline MAP@3")
print(f"Average MAP@3 Score: {round(np.mean(tfidf_scores), 4)}")

Q10: TF-IDF Pipeline MAP@3
Average MAP@3 Score: 0.2962
